# SESSION 2 - Prompt vs RAG vs Fine-Tuning

This notebook explores the core architectural trade-offs between Prompt Engineering, Retrieval-Augmented Generation (RAG), and Fine-Tuning. It features prompt creation for social media bios, a functional Python-based RAG simulation over local IPL statistics, comparative scenario mapping across popular Indian apps, a decision framework comparison table, and a production architecture evaluation for a BookMyShow movie recommendation chatbot.

### Task 1: Catchy Instagram Bio Generation Prompt

In [ ]:
bio_prompt = """
Role: You are a creative social media branding specialist.
Task: Generate 3 catchy, aesthetic Instagram bio options for a college student.

Student Profile:
- Major: Computer Science Engineering student in Ahmedabad
- Hobbies: Coding, weekend cafe hopping, and late-night gaming
- Personality: Chill, witty, and tech-curious

Constraints:
- Maximum 150 characters per bio
- Include clean line breaks and modern emojis
- Avoid cheesy cliches
""".strip()

print("=== PROMPT ===\n", bio_prompt)

In [ ]:
bio_output = """
Option 1:
CSE @ Ahmedabad 💻
Fuelled by cold brews & semicolons ☕✨
Debugging life one commit at a time.

Option 2:
Building algorithms by day, gaming by night 🎮⚡
Searching for Ahmedabad's best sourdough 🥐
CS '26 | Keep it simple.

Option 3:
Turning caffeine into clean code ☕👨‍💻
Tech explorer & weekend aesthetic chaser.
Drop in the lobby 🎯
""".strip()

print(" GENERATED INSTAGRAM BIOS \n", bio_output)

### Task 2: Simulated Retrieval-Augmented Generation (RAG) Workflow

In [ ]:
def retrieve_relevant_facts(query, file_path="ipl_stats.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        facts = [line.strip() for line in f if line.strip()]
    
    query_tokens = [word.lower() for word in query.replace("?", "").split()]
    matched_facts = []
    for fact in facts:
        fact_lower = fact.lower()
        matches = sum(1 for token in query_tokens if token in fact_lower)
        if matches > 0:
            matched_facts.append((matches, fact))
    
    matched_facts.sort(key=lambda x: x[0], reverse=True)
    return [item[1] for item in matched_facts[:2]]

In [ ]:
def rag_qa_bot(user_question, file_path="ipl_stats.txt"):
    retrieved_context = retrieve_relevant_facts(user_question, file_path)
    
    if not retrieved_context:
        context_str = "No relevant statistics found in database."
        grounded_answer = "I am sorry, but I do not have verified statistics to answer that question."
    else:
        context_str = "\n".join(f"- {fact}" for fact in retrieved_context)
        grounded_answer = f"Based on official records, {retrieved_context[0]}"
    
    rag_prompt = (
        f"=== RETRIEVED CONTEXT ===\n{context_str}\n\n"
        f"=== USER QUESTION ===\n{user_question}\n\n"
        f"=== SYNTHESIZED GROUNDED ANSWER ===\n{grounded_answer}"
    )
    return rag_prompt

In [ ]:
sample_query = "Who won the Orange Cap in IPL 2024 and how many runs were scored?"
result_pipeline = rag_qa_bot(sample_query)
print(result_pipeline)

In [ ]:
sample_query_2 = "What was the highest team score in IPL history?"
result_pipeline_2 = rag_qa_bot(sample_query_2)
print(result_pipeline_2)

### Task 3: Real-World Scenarios from Popular Apps (Prompting vs. RAG vs. Fine-Tuning)

1. **Prompt Engineering: Zomato Festive Seasonal Chatbot Persona**
   - *Scenario:* Zomato wants its customer-facing support chatbot to temporarily adopt a cheerful, festive Diwali or Navratri theme with Gujarati/Hindi greeting touches (*"Saal Mubarak!"*, festive food emoji flourishes).
   - *Why Prompting is Best:* This is a pure stylistic and behavioral adjustment that changes frequently. Updating a system prompt takes seconds, costs zero training dollars, and avoids touching model weights or external databases.

2. **Retrieval-Augmented Generation (RAG): Flipkart Product Technical Q&A**
   - *Scenario:* A customer on a specific smartphone product page asks: *"Does this specific variant support 5G carrier aggregation on Jio, and does the box include a 67W charger?"*
   - *Why RAG is Best:* Product inventories, seller bundle options, and technical specs change daily across millions of items. Fine-tuning cannot keep up with real-time catalog changes, and prompt engineering cannot fit millions of spec sheets in context. RAG dynamically searches the live product database for that exact SKU and supplies verifiable answers.

3. **Fine-Tuning: Paytm Regulatory Compliance & Suspicious Dispute Classification**
   - *Scenario:* Paytm must automatically parse customer transaction dispute descriptions and classify them into strict Reserve Bank of India (RBI) fraud categories (e.g., unauthorized SIM swap, vishing, QR code tampering) in audit-grade JSON.
   - *Why Fine-Tuning is Best:* The regulatory compliance taxonomy is fixed and highly specialized. Prompted models are prone to hallucinating categories or drifting under emotionally charged customer phrasing. Fine-tuning a smaller, private model directly bakes the immutable classification rules into model weights with sub-20ms latency and 100% deterministic output.

### Task 4: Comprehensive Comparison Table (Prompt Engineering vs. RAG vs. Fine-Tuning)

| Evaluation Criteria | Prompt Engineering | Retrieval-Augmented Generation (RAG) | Fine-Tuning |
| :--- | :--- | :--- | :--- |
| **Setup Cost** | **Lowest ($0 - negligible):** Requires no training compute, specialized GPU hardware, or external database infrastructure. | **Moderate:** Incurs expenses for vector database hosting (e.g., Pinecone, Chroma), embedding generation, and runtime LLM inference tokens. | **Highest ($$$):** Requires expensive high-end GPU compute clusters, dedicated data curation, hyperparameter tuning, and custom model hosting. |
| **Time to Deploy** | **Immediate (Minutes to Hours):** Developers simply iterate on system prompts, few-shot examples, and boundary rules in application code. | **Fast to Moderate (Days to Weeks):** Involves building ingestion pipelines, document chunking, embedding generation, vector indexing, and retriever tuning. | **Slow (Weeks to Months):** Requires collecting thousands of clean domain pairs, running training epochs, evaluating against benchmarks, and managing model drift. |
| **Maintenance Effort** | **Minimal:** Maintained via simple text updates to prompts; low operational overhead. | **Moderate:** Requires maintaining document sync pipelines, updating vector embeddings when source files change, and monitoring retrieval latency. | **High:** Requires continuous evaluation, retraining runs when data paradigms change, and managing versioned model weights. |
| **Ideal Use Cases** | Persona styling, creative copy generation, quick prototyping, translation, and structured summarization. | Dynamic knowledge bases, customer support with live policy files, e-commerce product Q&A, and legal document search. | Specialized industry jargon (medical/legal), strict proprietary JSON schemas, edge deployment of small SLMs, and fixed-domain classifiers. |

### Task 5: Architecture Decision for a BookMyShow-Style Movie Recommendation Chatbot

#### Advice Generated from ChatGPT:
> *"For a BookMyShow-style movie recommendation chatbot, the strongly recommended architecture is **Retrieval-Augmented Generation (RAG)**, potentially augmented with light Prompt Engineering for conversational tone.* 
> 
> *Key Reasons Provided:*
> *1. **Real-Time Data Velocity:** Movie schedules, city theatre locations, seat availability, ticket pricing, and critic ratings change multiple times a day. Fine-tuning is unfeasible because model weights cannot update dynamically without continuous re-training.* 
> *2. **Accuracy & Verifiability:** An LLM relying purely on internal memory will hallucinate showtimes or recommend movies that left theatres weeks ago. A RAG pipeline connects to the BookMyShow database to fetch active screening times, genre tags, and synopsis data.* 
> *3. **Implementation Speed:** RAG allows rapid deployment using off-the-shelf embedding models and an existing database without requiring high-cost machine learning training infrastructure."*

#### My Final Decision & Strategic Rationale:
I completely agree with the recommendation to adopt a **RAG-First Architecture**.

**Why RAG is the Superior Choice for BookMyShow:**
1. **Live Inventory Grounding:** Movie ticketing is a real-time transactional domain. A customer asking *"What IMAX 3D movies are showing in Ahmedabad after 7 PM tonight under ₹350?"* requires querying active theatre inventory. Fine-tuning a model on static movie plots would be useless for answering live screening logistics, whereas RAG retrieves exact real-time database records.
2. **Cost & Operational Efficiency:** BookMyShow's movie catalog updates every single Friday. Updating an indexed vector database takes fractions of a second with zero training cost, making RAG exponentially more scalable and maintainable than maintaining fine-tuned checkpoints.